# 02 · Entrenamiento base de los tres modelos

Objetivo: entrenar regresión logística, Random Forest y ResNet 1D con configuraciones fijas, sin búsqueda de hiperparámetros, para ambas etapas sobre las particiones guardadas en 01.

Proyecto de ECG con dos etapas: reposo/actividad y clasificación de cuatro ritmos. Ejecutar los cuadernos **01 → 02 → 03 → 04** con la misma carpeta de resultados. Estos archivos se entregan sin salidas ejecutadas: no contienen métricas inventadas.


## Preparación en Colab
Instalar dependencias, montar Drive y editar las rutas. Subir previamente los datos a Drive; no hace falta subir el ZIP de 2,5 GB si se usa el Parquet Chapman ya consolidado. Para entrenar ResNet, seleccionar un entorno con GPU. El código también admite CPU, con mayor tiempo de ejecución.

Referencias: [entrada y salida de archivos en Colab](https://colab.research.google.com/notebooks/io.ipynb) y [preguntas frecuentes](https://research.google.com/colaboratory/faq.html).


In [ ]:
%pip -q install pyarrow seaborn joblib scipy scikit-learn skl2onnx onnx


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

# EDITAR: carpeta del proyecto en tu Google Drive. Usar la misma en los 4 cuadernos.
PROJECT_DIR = Path('/content/drive/MyDrive/TP_IA')
REST_DIR = PROJECT_DIR / 'RestingClass' / 'data_resting'
ACTIVE_DIR = PROJECT_DIR / 'RestingClass' / 'data_not_resting'
CHAPMAN_FILE = PROJECT_DIR / 'Detector' / 'chapman_lead_I_consolidated.parquet'
STAGES = [1, 2]  # Usar [2] si primero quieres trabajar solamente con Chapman.
RESULTS = PROJECT_DIR / 'results_colab'
PREPARED = RESULTS / 'prepared'
MODELS = RESULTS / 'models'
FIGURES = RESULTS / 'figures'
for directory in (RESULTS, PREPARED, MODELS, FIGURES):
    directory.mkdir(parents=True, exist_ok=True)
assert STAGES and set(STAGES) <= {1, 2}, 'Elegir etapas 1 y/o 2.'
SEED = 42
print('Resultados:', RESULTS)


In [ ]:
import json, random, uuid, hashlib, gc, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import skew, kurtosis
from scipy.signal import find_peaks, welch
from scipy.interpolate import interp1d
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, StratifiedKFold, GridSearchCV, learning_curve
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, ConfusionMatrixDisplay, classification_report
from IPython.display import display

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASS_NAMES = {1: ['Reposo', 'Actividad'], 2: ['Bradicardia sinusal', 'Taquicardia sinusal', 'Fibrilación auricular', 'Ritmo sinusal normal']}
FEATURE_NAMES = {
    1: ['Kurtosis', 'ZCR', 'Crest_Factor', 'Skewness', 'HR', 'RMSSD', 'SDNN', 'EMG_Power', 'Baseline_Power', 'LF_HF_Ratio', 'Morph_Corr', 'Std_Dev'],
    2: ['hr', 'sdnn', 'rmssd', 'pnn50', 'pnn20', 'rr_cv', 'sd1', 'sd2', 'sd1_sd2', 'skew', 'kurt']
}
def save_json(filename, value):
    Path(filename).write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')

def load_prepared(stage):
    filename = PREPARED / f'stage{stage}.npz'
    if not filename.exists():
        raise FileNotFoundError(f'{filename}: ejecutar primero el cuaderno 01 para esta etapa.')
    with np.load(filename, allow_pickle=False) as stored:
        data = {key: stored[key] for key in stored.files}
    return data

def metrics(y, pred, probs):
    # Macro para comparar con igual peso todas las clases, en ambas etapas.
    auc = roc_auc_score(y, probs[:, 1]) if probs.shape[1] == 2 else roc_auc_score(y, probs, multi_class='ovr', average='macro')
    return dict(accuracy=accuracy_score(y, pred), precision_macro=precision_score(y, pred, average='macro', zero_division=0), recall_macro=recall_score(y, pred, average='macro', zero_division=0), f1_macro=f1_score(y, pred, average='macro', zero_division=0), auc=auc)

def make_cv(stage, y, groups, folds=3):
    cv = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=SEED) if stage == 1 else StratifiedKFold(n_splits=folds, shuffle=True, random_state=SEED)
    splits = list(cv.split(np.zeros((len(y), 1)), y, groups if stage == 1 else None))
    for tr, va in splits:
        if len(np.unique(y[tr])) != len(CLASS_NAMES[stage]) or len(np.unique(y[va])) != len(CLASS_NAMES[stage]):
            raise ValueError('Un fold no contiene todas las clases. Revisar sujetos/clases o reducir folds.')
        if stage == 1:
            assert not set(groups[tr]) & set(groups[va]), 'Sujetos compartidos entre folds.'
    return splits

def make_loader(signals, labels, batch_size, shuffle=False):
    # Normalización por señal; no ajusta estadísticas sobre otras muestras.
    a = np.asarray(signals, dtype=np.float32)
    a = (a - a.mean(axis=1, keepdims=True)) / (a.std(axis=1, keepdims=True) + 1e-8)
    dataset = TensorDataset(torch.from_numpy(a).unsqueeze(1), torch.as_tensor(labels, dtype=torch.long))
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        ce_loss = nn.CrossEntropyLoss(weight=self.alpha, reduction='none')(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


class BasicBlock1D(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(cin, cout, 5, stride=stride, padding=2, bias=False)
        self.bn1 = nn.BatchNorm1d(cout)
        self.conv2 = nn.Conv1d(cout, cout, 5, padding=2, bias=False)
        self.bn2 = nn.BatchNorm1d(cout)
        self.shortcut = nn.Sequential()
        if stride != 1 or cin != cout:
            self.shortcut = nn.Sequential(nn.Conv1d(cin, cout, 1, stride=stride, bias=False), nn.BatchNorm1d(cout))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        return F.relu(self.bn2(self.conv2(out)) + self.shortcut(x))

class ResNet1D(nn.Module):
    def __init__(self, classes, width, dropout):
        super().__init__()
        self.conv1 = nn.Conv1d(1, width, 7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(width)
        self.maxpool = nn.MaxPool1d(3, stride=2, padding=1)
        self.layer1 = nn.Sequential(BasicBlock1D(width, width), BasicBlock1D(width, width))
        self.layer2 = nn.Sequential(BasicBlock1D(width, width*2, 2), BasicBlock1D(width*2, width*2))
        self.layer3 = nn.Sequential(BasicBlock1D(width*2, width*4, 2), BasicBlock1D(width*4, width*4))
        self.layer4 = nn.Sequential(BasicBlock1D(width*4, width*8, 2), BasicBlock1D(width*8, width*8))
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(width*8, classes))
    def forward(self, x):
        x = self.maxpool(F.relu(self.bn1(self.conv1(x))))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        return self.fc(self.avgpool(x).flatten(1))

def predict_network(model, loader):
    model.eval()
    chunks = []
    with torch.no_grad():
        for signals, _ in loader:
            chunks.append(F.softmax(model(signals.to(DEVICE)), dim=1).cpu().numpy())
    return np.concatenate(chunks)

print('Dispositivo:', DEVICE)
versions = {name: __import__(name).__version__ for name in ['numpy', 'pandas', 'sklearn', 'scipy', 'torch', 'joblib']}
versions['python'] = platform.python_version()
save_json(RESULTS / 'environment.json', versions)
display(versions)


## Tres modelos base y métricas
Regresión logística: C=1. Random Forest: 100 árboles, sin límite de profundidad y hoja mínima 1. ResNet: arquitectura residual de los scripts, Adam con learning rate 0,001 y un presupuesto fijo de épocas configurable. Estos valores son referencias iniciales, no hiperparámetros seleccionados por búsqueda. Los pipelines clásicos ajustan el escalador solamente con entrenamiento. La red recibe señales normalizadas por ventana.

Se utiliza balance de clases en los modelos clásicos y FocalLoss ponderada en la red de etapa 1, como en el script original; la red de etapa 2 usa cross-entropy. F1 macro es la métrica principal; también se guardan accuracy, precision macro, recall macro y AUC. Las métricas corresponden a validación, usada también para seleccionar el checkpoint de la red; no constituyen una prueba final independiente.


In [ ]:
BASE_EPOCHS = {1: 10, 2: 10}  # Presupuesto fijo, sin early stopping ni búsqueda.
BASE_BATCH_SIZE = {1: 64, 2: 128}
assert all(isinstance(BASE_EPOCHS[s], int) and BASE_EPOCHS[s] > 0 for s in STAGES)
if DEVICE.type == 'cpu':
    print('ResNet se entrenará en CPU. Seleccionar GPU en Colab para acelerar.')


In [ ]:
rows = []
for stage in STAGES:
    data = load_prepared(stage)
    tr, va = data['train_idx'], data['val_idx']
    candidates = [
        ('baseline', 'Logistic Regression', 'lr', LogisticRegression(C=1.0, max_iter=2000, class_weight='balanced', random_state=SEED)),
        ('baseline_rf', 'Random Forest', 'rf', RandomForestClassifier(n_estimators=100, max_depth=None, min_samples_leaf=1, class_weight='balanced', random_state=SEED, n_jobs=-1))
    ]
    for artifact, name, step, classifier in candidates:
        baseline = Pipeline([('scaler', StandardScaler()), (step, classifier)])
        baseline.fit(data['features'][tr], data['y'][tr])
        probs = baseline.predict_proba(data['features'][va])
        preds = probs.argmax(axis=1)
        row = dict(stage=stage, model=name, variant='base', **metrics(data['y'][va], preds, probs))
        rows.append(row)
        joblib.dump(baseline, MODELS / f'stage{stage}_{artifact}.joblib')
        save_json(MODELS / f'stage{stage}_{artifact}_meta.json', dict(dataset_id=str(data['dataset_id']), classes=CLASS_NAMES[stage], threshold=0.5, model=name))
        print(f'Etapa {stage} — {name}')
        display(pd.DataFrame([row]))
        print(classification_report(data['y'][va], preds, target_names=CLASS_NAMES[stage], zero_division=0))
        pd.DataFrame(rows).to_csv(RESULTS / 'baseline_metrics.csv', index=False)
    del data
    gc.collect()


## ResNet 1D base
Esta celda entrena una red nueva por etapa y guarda artefactos con el sufijo `baseline_resnet`, separados de la red del cuaderno 03. El número de épocas es fijo; se conserva el checkpoint de menor pérdida de validación y se recalculan las predicciones después de cargarlo. El historial se guarda al finalizar cada época.


In [ ]:
for stage in STAGES:
    # Reiniciar semillas para que la inicialización de esta red sea reproducible.
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    data = load_prepared(stage)
    tr, va = data['train_idx'], data['val_idx']
    classes = len(CLASS_NAMES[stage])
    cfg = dict(classes=classes, width=16 if stage == 1 else 32, dropout=0.5 if stage == 1 else 0.0)
    model = ResNet1D(**cfg).to(DEVICE)
    train_loader = make_loader(data['raw'][tr], data['y'][tr], BASE_BATCH_SIZE[stage], shuffle=True)
    val_loader = make_loader(data['raw'][va], data['y'][va], BASE_BATCH_SIZE[stage])
    if stage == 1:
        counts = np.bincount(data['y'][tr], minlength=classes)
        weights = torch.tensor(len(tr)/(classes*counts), dtype=torch.float32, device=DEVICE)
        criterion = FocalLoss(alpha=weights, gamma=2.0)
    else:
        criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3 if stage == 1 else 1e-4)
    checkpoint = MODELS / f'stage{stage}_baseline_resnet.pth'
    best_loss, history = float('inf'), []
    for epoch in range(1, BASE_EPOCHS[stage]+1):
        model.train()
        total, count, train_true, train_pred = 0.0, 0, [], []
        for signals, labels in train_loader:
            signals, labels = signals.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            logits = model(signals)
            loss = criterion(logits, labels)
            loss.backward(); optimizer.step()
            total += loss.item()*len(labels); count += len(labels)
            train_true.extend(labels.cpu().tolist())
            train_pred.extend(logits.detach().argmax(1).cpu().tolist())
        model.eval()
        val_total, val_count = 0.0, 0
        with torch.no_grad():
            for signals, labels in val_loader:
                signals, labels = signals.to(DEVICE), labels.to(DEVICE)
                val_total += criterion(model(signals), labels).item()*len(labels)
                val_count += len(labels)
        probs = predict_network(model, val_loader)
        val_loss = val_total/val_count
        history.append(dict(epoch=epoch, train_loss=total/count, val_loss=val_loss, train_f1_macro=f1_score(train_true, train_pred, average='macro', zero_division=0), val_f1_macro=f1_score(data['y'][va], probs.argmax(1), average='macro', zero_division=0)))
        pd.DataFrame(history).to_csv(RESULTS / f'stage{stage}_baseline_resnet_history.csv', index=False)
        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), checkpoint)
            save_json(MODELS / f'stage{stage}_baseline_resnet_meta.json', dict(dataset_id=str(data['dataset_id']), config=cfg, best_epoch=epoch, best_val_loss=best_loss, budget_epochs=BASE_EPOCHS[stage], threshold=0.5, classes=CLASS_NAMES[stage]))
        print(f'Etapa {stage}, ResNet base, época {epoch}: val_loss={val_loss:.4f}')
    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True))
    probs = predict_network(model, val_loader)
    preds = probs.argmax(axis=1)
    row = dict(stage=stage, model='ResNet1D', variant='base', **metrics(data['y'][va], preds, probs))
    rows.append(row)
    print(f'Etapa {stage} — ResNet1D base (mejor checkpoint)')
    display(pd.DataFrame([row]))
    print(classification_report(data['y'][va], preds, target_names=CLASS_NAMES[stage], zero_division=0))
    pd.DataFrame(rows).to_csv(RESULTS / 'baseline_metrics.csv', index=False)
    del data, model, train_loader, val_loader
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()


## Comparación final de los tres modelos
Con ambas etapas seleccionadas, la tabla debe contener seis filas: tres modelos por etapa. Las cifras se calculan al ejecutar y se guardan juntas en `baseline_metrics.csv`.


In [ ]:
summary = pd.DataFrame(rows).sort_values(['stage', 'model']).reset_index(drop=True)
assert len(summary) == 3*len(STAGES), 'Falta ejecutar alguna celda de entrenamiento.'
summary.to_csv(RESULTS / 'baseline_metrics.csv', index=False)
display(summary)
display(summary.pivot(index='model', columns='stage', values='f1_macro').rename_axis(columns='Etapa'))
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=summary, x='model', y='f1_macro', hue='stage', errorbar=None, ax=ax)
ax.set(ylim=(0,1), ylabel='F1 macro de validación', title='Comparación de los tres modelos base')
fig.tight_layout(); fig.savefig(FIGURES / 'baseline_model_comparison.png', dpi=150); plt.show(); plt.close(fig)


## Interpretación
Anotar qué clases reconoce mejor cada modelo y por qué accuracy puede ocultar errores en clases minoritarias. En 03 se ajustan los hiperparámetros de los modelos clásicos y se entrena la red con mayor presupuesto, scheduler y early stopping; la comparación usa las mismas particiones. La ResNet base y la de 03 no reciben una búsqueda Grid Search.
